In [ ]:
from torch.utils.data import DataLoader, TensorDataset, random_split

from model      import ConvAttnPool
from metrics    import eval_model, compare_metric
from evaluation import all_metrics
import numpy     as np

from functools import partial
import os
import tempfile
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from ray.tune.schedulers import ASHAScheduler
from ray import tune, train
import ray.cloudpickle as pickle
# from ray.train import Checkpoint, get_checkpoint

In [9]:
def load_data(data_dir="./data"):
    X_train = np.load('/home/drew/FL-with-MIMIC/Replicating Mullenbach/X_train.npy')
    X_test  = np.load('/home/drew/FL-with-MIMIC/Replicating Mullenbach/X_test.npy')

    Y_train = np.load('/home/drew/FL-with-MIMIC/Replicating Mullenbach/Y_train.npy')
    Y_test  = np.load('/home/drew/FL-with-MIMIC/Replicating Mullenbach/Y_test.npy')

    X_train = torch.from_numpy(X_train).long()
    Y_train = torch.from_numpy(Y_train).type(torch.float32)

    X_test  = torch.from_numpy(X_test) .long()
    Y_test  = torch.from_numpy(Y_test) .type(torch.float32)
    trainset = TensorDataset(X_train,Y_train)
    testset  = TensorDataset(X_test ,Y_test )
    return trainset, testset

In [ ]:
def train_model(config, data_dir=None):
    net = ConvAttnPool(num_of_filters = config["l1"], 
                        kernel_size    = config["l2"])

    device = "cpu"
    if torch.cuda.is_available():
        device = torch.device('cuda')
    net.to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.SGD(net.parameters(), lr=config["lr"], momentum=0.9)

    trainset, testset = load_data(data_dir)

    test_abs = int(len(trainset) * 0.8)
    train_subset, val_subset = random_split(
        trainset, [test_abs, len(trainset) - test_abs]
    )

    trainloader = torch.utils.data.DataLoader(
        train_subset, batch_size=int(config["batch_size"]), shuffle=True, num_workers=8
    )
    valloader = torch.utils.data.DataLoader(
        val_subset, batch_size=int(config["batch_size"]), shuffle=True, num_workers=8
    )

    # 
    for epoch in range(0, 200):
        running_loss = 0.0
        epoch_steps = 0
        for i, data in enumerate(trainloader, 0):
            # get the inputs; data is a list of [inputs, labels]
            inputs, labels = data
            inputs, labels = inputs.to(device), labels.to(device)

            preds, alpha = net(inputs)
            preds = preds.type(torch.float32).to(device)
            
            loss  = criterion(preds, labels)

            # Zero the gradients
            optimizer.zero_grad()
            loss     .backward()
            optimizer.step()

            running_loss += loss.item()
            epoch_steps += 1
            if i % 2000 == 1999:  # print every 2000 mini-batches
                print(
                    "[%d, %5d] loss: %.3f"
                    % (epoch + 1, i + 1, running_loss / epoch_steps)
                )
                running_loss = 0.0

    # Validation loss
    val_loss = 0.0
    val_steps = 0
    total = 0
    correct = 0
    for i, data in enumerate(valloader,0):
        with torch.no_grad():
            inputs, labels = data
            inputs, labels = inputs.to(device), labels.to(device)

            preds, _    = net(inputs)
            pred_labels = (F.sigmoid(preds) >= 0.5).type(torch.float32)

            loss = criterion(pred_labels, labels)
            val_loss += loss.cpu().numpy()
            val_steps += 1

    tune.report({"loss": val_loss / val_steps, "net_state_dict": net.state_dict(),})

    print("Finished Training")

In [11]:
def test_accuracy(model,device = 'cpu'):
    model.eval()
    trainset, testset = load_data()

    testloader = torch.utils.data.DataLoader(
        testset, batch_size=4, shuffle=False, num_workers=2
    )

    all_pred     = torch.empty((0,), dtype = torch.float32).to(device)
    all_labels   = torch.empty((0,), dtype = torch.float32).to(device)
    all_pred_raw = torch.empty((0,), dtype = torch.float32).to(device)

    for data_inputs, data_labels in testloader:
        data_inputs = data_inputs.to(device)
        data_labels = data_labels.to(device)
        preds, _    = model(data_inputs)
        pred_labels = (F.sigmoid(preds) >= 0.5).long()

        all_pred     = torch.cat((all_pred,pred_labels), dim = 0 )
        all_labels   = torch.cat((all_labels,data_labels), dim = 0)
        all_pred_raw = torch.cat([all_pred_raw, preds], dim = 0)

    all_met = (all_metrics(yhat =   all_pred.cpu().detach().numpy(), y = all_labels.cpu().detach().numpy(), yhat_raw=all_pred_raw.cpu().detach().numpy()  ))
    return all_met['auc_macro']

In [12]:
num_samples    = 50; # trials
max_num_epochs = 350; # Max epochs
data_dir = os.path.abspath("./data")
load_data(data_dir)
config = {
    "l1": tune.choice([i for i in range(15,20 + 1)]), # Num of Filters
    "l2": tune.choice([i for i in range(3 ,10 + 1)]), # Filtern size
    "lr": tune.loguniform(0.0001, 0.1),
    "batch_size": tune.choice([8, 16, 32, 64]),
}
scheduler = ASHAScheduler(
    metric="loss",
    mode="min",
    max_t=max_num_epochs,
    grace_period=100,
    reduction_factor=2,
)
result = tune.run(
    partial(train_model, data_dir=data_dir),
    resources_per_trial={"cpu": 4, "gpu": 1},
    config=config,
    num_samples=num_samples,
    scheduler=scheduler,
)

2025-08-22 13:02:56,962	INFO tune.py:616 -- [output] This uses the legacy output and progress reporter, as Jupyter notebooks are not supported by the new engine, yet. For more information, please see https://github.com/ray-project/ray/issues/36949


Trial name,loss,should_checkpoint
train_model_52b1d_00000,0.685898,True
train_model_52b1d_00001,0.685605,True


2025-08-22 13:22:41,151	WARNING tune.py:219 -- Stop signal received (e.g. via SIGINT/Ctrl+C), ending Ray Tune run. This will try to checkpoint the experiment state one last time. Press CTRL+C (or send SIGINT/SIGKILL/SIGTERM) to skip. 
2025-08-22 13:22:41,167	INFO tune.py:1009 -- Wrote the latest version of all result files and experiment state to '/home/drew/ray_results/train_model_2025-08-22_13-02-56' in 0.0157s.
2025-08-22 13:22:43,440	INFO tune.py:1041 -- Total run time: 1186.48 seconds (1184.18 seconds for the tuning loop).
2025-08-22 13:22:43,442	WARNING tune.py:1056 -- Experiment has been interrupted, but the most recent state was saved.
Resume experiment with: tune.run(..., resume=True)
2025-08-22 13:22:43,465	WARNING experiment_analysis.py:180 -- Failed to fetch metrics for 48 trial(s):
- train_model_52b1d_00002: FileNotFoundError('Could not fetch metrics for train_model_52b1d_00002: both result.json and progress.csv were not found at /home/drew/ray_results/train_model_2025-08-

In [ ]:
best_trial = result.get_best_trial("loss", "min", "last")
print(f"Best trial config: {best_trial.config}")
print(f"Best trial final validation loss: {best_trial.last_result['loss']}")


NameError: name 'result' is not defined

In [ ]:
# print(f"Best trial final validation accuracy: {best_trial.last_result['accuracy']}")
best_trained_model = ConvAttnPool(num_of_filters = best_trial.config["l1"], 
                                  kernel_size    = best_trial.config["l2"])

device = "cpu"
if torch.cuda.is_available():
    device = torch.device("cuda")
best_trained_model.to(device)

best_checkpoint = result.get_best_checkpoint(trial=best_trial, mode="max") # , metric="accuracy"
with best_checkpoint.as_directory() as checkpoint_dir:
    print(f'Loading from {checkpoint_dir}')
    data_path = Path(checkpoint_dir) / "data.pkl"
    with open(data_path, "rb") as fp:
        best_checkpoint_data = pickle.load(fp)

    best_trained_model.load_state_dict(best_checkpoint_data["net_state_dict"])
    test_acc = test_accuracy(best_trained_model, device)
    print("Best trial test set accuracy: {}".format(test_acc))